In [ ]:
# Prerequisite boilerplate
import shutil
from pathlib import Path

ROOT = Path("../data/tiny-imagenet")
SEED = 5504132


def splits_file_name() -> Path:
    return ROOT / "splits" / f"tiny-{TRAIN_FRACTION}-{1 - TRAIN_FRACTION}_{SEED}"

# Tiny ImageNet splits: 10 classes + other

An 11-way classification task: source classes `000`–`009` keep labels `0`–`9`, and the
other 190 become label `10` (`other`).

- **train / val:** the downloaded `train/` folder, split 70/30 within each source class.
- **test:** the downloaded `test/` folder (Tiny ImageNet's official validation set),
  kept for final reporting only (PLAN.md D14).

Writes CSVs to `OUT`, which is rebuilt on every run: `{train,val,test}.csv`
(`sample_id,path,label`, paths relative to `ROOT`), `class_to_idx.csv` and
`split.csv`.

In [ ]:
import pandas as pd

TRAIN_FRACTION = 0.7
KEEP = ["000", "001", "002", "003", "004", "005", "006", "007", "008", "009"]

OUT = (
    ROOT
    / "splits"
    / f"top{len(KEEP)}-other_{TRAIN_FRACTION}-{1 - TRAIN_FRACTION}_seed{SEED}"
)
shutil.rmtree(OUT, ignore_errors=True)

class_to_idx = {c: i for i, c in enumerate(KEEP)} | {"other": len(KEEP)}

## Split

In [ ]:
def files(split: str) -> pd.DataFrame:
    # Sorted so the seeded sample below sees the same order on every platform.
    paths = sorted(p.relative_to(ROOT) for p in (ROOT / split).glob("*/*"))
    return pd.DataFrame(
        {
            "sample_id": [p.with_suffix("").as_posix() for p in paths],
            "path": [p.as_posix() for p in paths],
            "source": [p.parent.name for p in paths],
            "label": [class_to_idx.get(p.parent.name, len(KEEP)) for p in paths],
        }
    )


pool = files("train")
train = pool.groupby("source").sample(frac=TRAIN_FRACTION, random_state=SEED)
splits = {
    "train": train.sort_index(),
    "val": pool.drop(train.index),
    "test": files("test"),
}

## Write

In [ ]:
OUT.mkdir(parents=True)
for name, df in splits.items():
    df[["sample_id", "path", "label"]].to_csv(OUT / f"{name}.csv", index=False)

pd.Series(class_to_idx, name="idx").rename_axis("class").to_csv(
    OUT / "class_to_idx.csv"
)
pd.Series(
    {
        "root": ROOT.resolve().name,
        "seed": SEED,
        "train_fraction": TRAIN_FRACTION,
        "holdout": "per source class, from train/",
        "test": "test/ (official Tiny ImageNet validation set)",
    }
    | {f"{name}_size": len(df) for name, df in splits.items()},
    name="value",
).rename_axis("key").to_csv(OUT / "split.csv")

## Check

Read back what was written: one class map, disjoint splits, every source class split 70/30. The table is images per label.

In [ ]:
written = pd.concat(
    [pd.read_csv(OUT / f"{n}.csv").assign(split=n) for n in splits], ignore_index=True
)
classes = pd.read_csv(OUT / "class_to_idx.csv", dtype={"class": str})
assert dict(zip(classes["class"], classes["idx"], strict=True)) == class_to_idx

assert written["sample_id"].is_unique, "a sample is in two splits"
assert written["label"].between(0, len(KEEP)).all()
assert all((ROOT / p).is_file() for p in written["path"])

n = pd.crosstab(written["path"].str.split("/").str[1], written["split"])
assert (n["train"] == (TRAIN_FRACTION * (n["train"] + n["val"])).round()).all()

pd.crosstab(written["label"], written["split"])

## Class balance

`other` is 95% of every split (66,500 train images against 350 per kept class). The
manifests keep every image; balance is handled downstream:

- **train:** a `WeightedRandomSampler`, or class weights in the cross-entropy loss;
- **evaluate:** macro-averaged metrics (balanced accuracy, macro F1, per-class recall).
  Plain accuracy is 95% for a model that always predicts `other`.

## Load

A plain torch `Dataset` over a manifest, yielding the batch format of PLAN.md §3.1.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision.io import ImageReadMode, decode_image


class ManifestDataset(Dataset):
    def __init__(self, root: Path, manifest: Path):
        self.root = root
        self.rows = pd.read_csv(manifest).to_dict("records")

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int) -> dict:
        row = self.rows[i]
        image = decode_image(str(self.root / row["path"]), mode=ImageReadMode.RGB)
        return {
            "inputs": {"image": image.float() / 255},
            "targets": {"class": row["label"]},
            "meta": {"sample_id": row["sample_id"]},
        }


batch = next(iter(DataLoader(ManifestDataset(ROOT, OUT / "train.csv"), batch_size=8)))
batch["inputs"]["image"].shape, batch["targets"]["class"]